In [24]:
! open .

In [33]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from openai import OpenAI
import os

client = OpenAI()

# Set up your OpenAI API k
# Path to your PDF file
pdf_file = '/Users/markbotterill/code/idinsight/OLD_survey_designer/data/finalized_surveys/2020 July LFS APIS Questionnaire.pdf'  # Replace with the path to your PDF file

# Directory to store temporary images
image_folder = 'pdf_images'
os.makedirs(image_folder, exist_ok=True)

# Step 1: Extract text from PDF using OCR
print("Extracting text from PDF pages...")
pages = convert_from_path(pdf_file)
text_per_page = []

for page_number, page in enumerate(pages, start=1):
    image_path = os.path.join(image_folder, f'page_{page_number}.png')
    page.save(image_path, 'PNG')
    
    # Apply OCR to extract text
    text = pytesseract.image_to_string(Image.open(image_path))
    text_per_page.append({
        'page_number': page_number,
        'text': text
    })    
    print(f'Processed page {page_number}')

# Optional: Clean up images (uncomment if you want to delete images after processing)
# import shutil
# shutil.rmtree(image_folder)

# Step 2: Attempt to find an index/summary in the first few pages
print("\nAnalyzing the first few pages for an index or summary...")
first_pages_text = ' '.join([page['text'] for page in text_per_page[:3]])  # Adjust number of pages as needed

prompt_index = f"""
You are an assistant that summarizes documents.

The following text is from the beginning of a survey document:

\"\"\"
{first_pages_text}
\"\"\"

Does this text contain an index or summary of the modules included in the survey? 

- If yes, please list and summarize the modules.
- If no, respond with "Not found".
"""

response_index = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an expert document analyst."},
        {"role": "user", "content": prompt_index}
    ],
    max_tokens=500,
    temperature=0.0
)

index_summary = response_index.choices[0].message.content
print("\nIndex/Summary of Modules:\n", index_summary)

# Step 3: Recursively parse the document to detect topic changes
print("\nDetecting topics for each page and identifying modules...")

def detect_topic(text):
    prompt = f"""
The following text is extracted from a PDF with OCR, please
do your utmost to detect a common topic in the provided page text. 


\"\"\"
{text}
\"\"\"

REPLY ONLY WITH THE 5-10 words that describe the topic
"""
    # print(f"PRINTING PROMPT PASSED TO LLM {prompt}")
    response = client.chat.completions.create(
    model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert in topic detection."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=10,
        temperature=0.3
    )
    topic = response.choices[0].message.content.strip()
    return topic

def compare_topics(topic1, topic2):
    prompt = f"""
Are the following two topics loosely about the same area or are they quite 
different? Respond with "Same" or "Different".

Topic 1: {topic1}
Topic 2: {topic2}
"""
    response = client.chat.completions.create(
    model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a topic comparison assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=10,
        temperature=0
    )
    result = response.choices[0].message.content.strip()
    # print(f"HERE IS THE RESULT: {result}")
    return result

# Detect topics per page
for page in text_per_page:
    page['topic'] = detect_topic(page['text'])
    print(f"Page {page['page_number']}: Topic - {page['topic']}")

# Identify modules based on topic changes
modules = []
current_module = {
    'start_page': text_per_page[0]['page_number'],
    'end_page': text_per_page[0]['page_number'],
    'topic': text_per_page[0]['topic']
}

for i in range(1, len(text_per_page)):
    previous_topic = current_module['topic']
    current_topic = text_per_page[i]['topic']
    
    comparison = compare_topics(previous_topic, current_topic)
    
    if comparison == 'Same':
        # Extend current module
        current_module['end_page'] = text_per_page[i]['page_number']
    else:
        # Save the completed module
        modules.append(current_module)
        # Start a new module
        current_module = {
            'start_page': text_per_page[i]['page_number'],
            'end_page': text_per_page[i]['page_number'],
            'topic': current_topic
        }

# Add the last module
modules.append(current_module)

# List detected modules
print("\nDetected Modules:")
for module in modules:
    print(f"Module: {module['topic']}, Pages: {module['start_page']}-{module['end_page']}")

# Step 4: Create a document summary
print("\nGenerating a summary of the entire document...")

# Compile full text
full_text = ' '.join([page['text'] for page in text_per_page])

prompt_summary = f"""
Provide a concise summary (150 words or fewer) of the following survey document:

\"\"\"
{full_text}
\"\"\"
"""

response_summary = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are an expert document summarizer."},
        {"role": "user", "content": prompt_summary}
    ],
    max_tokens=200,
    temperature=0.5
)

document_summary = response_summary.choices[0].message.content.strip()
print("\nDocument Summary:\n", document_summary)

# Step 5: Generate module summaries and enrich them
print("\nCreating summaries for each module and enriching them...")

module_details = []

for module in modules:
    # Collect text for the module
    module_texts = [
        page['text'] 
        for page in text_per_page 
        if module['start_page'] <= page['page_number'] <= module['end_page']
    ]
    module_text = ' '.join(module_texts)
    
    # Step 1: Create a 5-10 word summary
    prompt_module_summary = f"""
Summarize the main topic of the following text in 5-10 words:

\"\"\"
{module_text}
\"\"\"
"""
    response_module_summary = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a concise summarizer."},
            {"role": "user", "content": prompt_module_summary}
        ],
        max_tokens=15,
        temperature=0.3
    )
    short_summary = response_module_summary.choices[0].message.content.strip()
    
    # Step 2: Enrich the summary with additional information
    prompt_enrich = f"""
We have detected that there is a module on "{short_summary}". 

Take the pages from the document labeled as being this topic and add some additional information that differentiates this module from just being a "{short_summary}".
"""
    response_enrich = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are an expert content enhancer."},
            {"role": "user", "content": prompt_enrich}
        ],
        max_tokens=100,
        temperature=0.5
    )
    enriched_info = response_enrich.choices[0].message.content.strip()
    
    # Save module details
    module_details.append({
        'module': short_summary,
        'pages': f"{module['start_page']}-{module['end_page']}",
        'additional_info': enriched_info
    })

# Display module summaries and additional information
print("\nModule Summaries and Additional Information:")
for detail in module_details:
    print(f"Module ({detail['pages']}): {detail['module']}")
    print(f"Additional Information: {detail['additional_info']}\n")

# Step 6: Customizable logic on embeddings (optional)
print("Generating embeddings for each module...")

# Example: Generate embeddings for each module's text
module_embeddings = []

for module in modules:
    module_texts = [
        page['text'] 
        for page in text_per_page 
        if module['start_page'] <= page['page_number'] <= module['end_page']
    ]
    module_text = ' '.join(module_texts)
    
    # Generate embedding
    response_embedding = client.embeddings.create(
        input=module_text,
        model="text-embedding-ada-002"
    )
    embedding_vector = response_embedding.data[0].embedding
    
    module_embeddings.append({
        'module': module['topic'],
        'embedding': embedding_vector
    })

print("Embeddings generated for modules")

Extracting text from PDF pages...
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Processed page 5
Processed page 6
Processed page 7
Processed page 8
Processed page 9
Processed page 10
Processed page 11
Processed page 12
Processed page 13
Processed page 14
Processed page 15
Processed page 16
Processed page 17

Analyzing the first few pages for an index or summary...

Index/Summary of Modules:
 Not found.

Detecting topics for each page and identifying modules...
Page 1: Topic - Philippine Labor Force and Poverty Surveys
Page 2: Topic - Demographic characteristics and education levels survey.
Page 3: Topic - Employment and economic characteristics of individuals
Page 4: Topic - Economic characteristics of employment and payment types.
Page 5: Topic - Employment status and job search characteristics
Page 6: Topic - Economic characteristics and employment status survey.
Page 7: Topic - Educational assistance programs and funding options
Page 8: Topic - Health status, n